## Init

In [ ]:
import numpy as np
import scipy.constants as phy_const
import matplotlib.pyplot as plt
import pickle

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import os
import pandas as pd
import pickle
import glob
import sys
import configparser

from matplotlib import rcParams as rc
rc["mathtext.fontset"] = "stix"
rc["font.family"] = "STIXGeneral"

# Enable LaTeX rendering
plt.rcParams['text.usetex'] = True
plt.rc("text", usetex=True)


In [ ]:
def compute_phi(P, Current):
    def cumtrapz(y, d):
        return np.concatenate((np.zeros(1), np.cumsum(d * (y[1:] + y[:-1]) / 2.0)))
    
    E   = compute_E(P, Current)
    phi = V - Current * Rext - cumtrapz(E, d=Delta_x)  # Discharge electrostatic potential
    return phi
    
def compute_E(P, Current):

    def compute_Kel(Te):
        # Xenon
        a =  6.25621116e-14
        b = -4.30874715e+00
        c = -1.45152836e+01
        d = -1.49229954e+01
        e = -5.68651763e+00
        f = 3.36357165e-01

        t = (1/Te)

        return 16./3.*a*t**f*np.exp(-b*t+ c*t**2 - d*t**3. + e*t**4)
    
    def trapz(y, d):
        return np.sum( (y[1:] + y[:-1]) )*d/2.0
    def gradient(y, d):
        dp_dz = np.empty_like(y)
        dp_dz[1:-1] = (y[2:] - y[:-2]) / (2 * d)
        dp_dz[0] = 2 * dp_dz[1] - dp_dz[2]
        dp_dz[-1] = 2 * dp_dz[-2] - dp_dz[-3]

        return dp_dz

    # TODO: This is already computed! Maybe move to the source
    #############################################################
    #       We give a name to the vars to make it more readable
    #############################################################
    ng = P[0,:]
    ni = P[1,:]
    ui = P[2,:]
    Te = P[3,:]
    ve = P[4,:]
    Gamma_i = ni*ui
    wce     = phy_const.e*B/m              # electron cyclotron frequency
    
    #############################
    #       Compute the rates   #
    #############################
    Kel = compute_Kel(Te)  # Electron - neutral  collision rate     TODO: Replace by good one

    ############################
    #       Wall Collisions    # 
    ############################     
    sigma      = 0.207*Te**(0.549)
    sigma_scl  = 1. - 8.3*np.sqrt(m/M)
    sigma[sigma > sigma_scl] = sigma_scl
    nu_iw      = 2 * 0.5 * (1.0 / (R2 - R1)) * np.sqrt(phy_const.e * Te / M)
    index_L0 = np.argmax(x_center > L0)
    nu_iw[index_L0:] = 0.0
    nu_ew      =  nu_iw / (1 - sigma)                                        # Electron - wall collision rate

    # TODO: OLD
    # sigma_old = 2.0 * Te / Estar  # SEE yield
    # sigma_old[sigma_old > 0.986] = 0.986
    # nu_iw = (
    #     (4.0 / 3.0) * (1.0 / (R2 - R1)) * np.sqrt(phy_const.e * Te / M)
    # )  # Ion - wall collision rate d
    # # Limit the collisions to inside the thruster
    # index_L0 = np.argmax(x_center > L0)
    # nu_iw[index_L0:] = 0.0

    alpha_B = (np.ones(NBPOINTS) * alpha_B1)                                                # Anomalous transport coefficient inside the thruster
    alpha_B = np.where(x_center < L0, alpha_B, alpha_B2)                                    # Anomalous transport coefficient in the plume
    alpha_B_smooth = np.copy(alpha_B)

    # smooth between alpha_B1 and alpha_B2
    for index in range(10, NBPOINTS - 9):
        alpha_B_smooth[index] = np.mean(alpha_B[index-10:index+10])
    alpha_B = alpha_B_smooth

    nu_m   = ng*Kel + alpha_B*wce + nu_ew                          # Electron momentum - transfer collision frequency
    
    mu_eff = (phy_const.e/(m*nu_m))*(1./(1 + (wce/nu_m)**2))       # Effective mobility
    dp_dz  = gradient(ni*Te, d = Delta_x)
    
    I0 = Current/(phy_const.e*A0)
    
    E = (I0 - Gamma_i) / (mu_eff * ni) - dp_dz / ni  # Discharge electric field
    return E


In [ ]:
##########################################################
#           POST-PROC PARAMETERS
##########################################################
Results     = "Results_SPT_1/test_21/"
# Results     = "Results_SPT_1/case_5v5_mgs/"
# Results     = "Results_SPT_1/case_5v0_mgs/"
Results     = "Results_SPT_h03/test_5v5/"
Results     = "Results_SPT_h03_1/test_6/"

PLOT_VARS   = True
ResultConfig = Results+'/Configuration.cfg'


In [ ]:

##########################################################
#           CONFIGURE PHYSICAL PARAMETERS
##########################################################

configFile = ResultConfig
config = configparser.ConfigParser()
config.read(configFile)

physicalParameters = config['Physical Parameters']

VG       = float(physicalParameters['Gas velocity'])                 # Gas velocity
M        = float(physicalParameters['Ion Mass'])*phy_const.m_u       # Ion Mass
m        = phy_const.m_e                                             # Electron mass
R1       = float(physicalParameters['Inner radius'])                 # Inner radius of the thruster
R2       = float(physicalParameters['Outer radius'])                 # Outer radius of the thruster
A0       = np.pi * (R2 ** 2 - R1 ** 2)                               # Area of the thruster
LENGTH   = float(physicalParameters['Length of axis'])               # length of Axis of the simulation
L0       = float(physicalParameters['Length of thruster'])           # length of thruster (position of B_max)
alpha_B1 = float(physicalParameters["Anomalous transport alpha_B1"])  # Anomalous transport
alpha_B2 = float(physicalParameters["Anomalous transport alpha_B2"])  # Anomalous transport
mdot     = float(physicalParameters['Mass flow'])                    # Mass flow rate of propellant
Te_Cath  = float(physicalParameters['Temperature Cathode'])          # Electron temperature at the cathode
Rext     = float(physicalParameters['Ballast resistor'])             # Resistor of the ballast
V0       = float(physicalParameters['Voltage'])                      # Potential difference
Estar    = float(physicalParameters['Crossover energy'])             # Crossover energy

##########################################################
#           NUMERICAL PARAMETERS
##########################################################
NumericsConfig = config['Numerical Parameteres']

NBPOINTS  = int(NumericsConfig['Number of points'])             # Number of cells
SAVERATE  = int(NumericsConfig['Save rate'])                    # Rate at which we store the data
CFL       = float(NumericsConfig['CFL'])                        # Nondimensional size of the time step
TIMEFINAL = float(NumericsConfig['Final time'])                 # Last time of simulation
Results   = NumericsConfig['Result dir']                        # Name of result directory
TIMESCHEME = NumericsConfig['Time integration']                        # Name of result directory

Delta_x  = LENGTH/NBPOINTS

##########################################################
#           Make the plots
#########################################################

# plt.style.use('classic')
# plt.rcParams["font.family"] = 'serif'
# plt.rcParams["font.weight"] = 'normal'
# plt.rcParams['figure.facecolor'] = 'white'
# plt.rcParams["font.size"] = 15
# plt.rcParams["lines.linewidth"] = 2

ResultsFigs = Results+"/Figs"
ResultsData = Results+"/Data"
if not os.path.exists(ResultsFigs):
    os.makedirs(ResultsFigs)

# open all the files in the directory and sort them to do the video in order
files       = glob.glob(ResultsData + "/*.pkl")
filesSorted = sorted(files, key = lambda x: os.path.getmtime(x), reverse=True)
files.sort(key=os.path.getmtime)
files = files[:]


In [ ]:

Current = np.zeros(np.shape(files)[0])
Voltage = np.zeros(np.shape(files)[0])
time    = np.zeros(np.shape(files)[0])


#####################################
#           Plot variables
#####################################

for i_save, file in enumerate(files):
    
    with open(file, 'rb') as f:
        [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)
    
    # Save the current
    Current[i_save] = J
    Voltage[i_save] = V
    time[i_save]    = t

#####################################
#           Plot current
#####################################


In [ ]:
%matplotlib widget
f, ax = plt.subplots(figsize=(8,3))
ax.plot(time/1e-3, Current)
ax.set_xlabel(r'$t$ [ms]', fontsize=18, weight = 'bold')
ax.set_ylabel(r'Current [A]', fontsize=18)
ax_V=ax.twinx()
ax_V.plot(time/1e-3, Voltage,'r')
ax_V.plot(time/1e-3, Voltage - Current * 10,'darkred')
ax_V.set_ylabel(r'Voltage [V]', fontsize=18)
ax.grid(True)
ax.set_title('Current and Voltage', fontsize=18)
# ax.set_xlim(0, time[-1]/1e-3)
ax.set_ylim(0, 1.1*Current.max())
ax.set_xlim(0, 2)
ax_V.set_ylim(0, 1.1*Voltage.max())
ax.tick_params(axis='y', labelcolor='black')
plt.tight_layout()
plt.savefig(ResultsFigs+"/Current.pdf", bbox_inches='tight')
ax.axvline(x=time[-1000]/1e-3, color='k', linestyle='--', label='L0')
ax.axhline(y=np.mean(Current[-1000:]), color='k', linestyle='--', label='Mean current')
print("Mean current: ", np.mean(Current[-1000:]))

In [ ]:
for i_save, file in enumerate(files[-1:]):

    # print("Preparing plot for i = ", i_save)
    #
    file_number = files.index(file)
    print(f"Processing file number: {file_number}")

    with open(file, "rb") as f:
        [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)

    E = compute_E(P, J)
    phi = compute_phi(P, J)

    fig = plt.figure(figsize=(8, 7))
    gls = fig.add_gridspec(4, 2)
    ax1 = fig.add_subplot(gls[0, 0])
    ax2 = fig.add_subplot(gls[0, 1])
    ax3 = fig.add_subplot(gls[1, 0])
    ax4 = fig.add_subplot(gls[1, 1])
    ax5 = fig.add_subplot(gls[2, 0])
    ax6 = fig.add_subplot(gls[2, 1])
    ax7 = fig.add_subplot(gls[3, :])

    ax = [ax1, ax2, ax3, ax4, ax5, ax6, ax7]

    ax_b = ax[0].twinx()
    ax[0].plot(x_center * 100, P[0, :], linewidth=1.8, markersize=3)
    ax_b.plot(x_center * 100, B, "r:", linewidth=0.9, markersize=3)
    ax_b.set_yticks([0, max(B)])
    # ax_b.set_ylabel(r'$B$ [T]', color='r')
    # ax[0].set_xlabel(r'$x~[m]$')
    ax[0].set_ylabel(r"$n_g$ [m$^{-3}$]")
    # ax[0].xaxis.set_tick_params(which='both', size=0, width=1.5, labelsize=0)
    ax[0].set_xticklabels([])
    ax[0].yaxis.set_tick_params(which="both", size=5, width=1.5, labelsize=13)
    # ax[0].legend(loc = 'lower center', fontsize = 12)
    ax[0].set_ylim([0, max(P[0, :]) * 1.1])

    ax_phi = ax[1].twinx()
    ax[1].plot(x_center * 100, E, linewidth=1.8, markersize=3)
    ax_phi.plot(x_center * 100, phi, color="r", linewidth=1.8, markersize=3)
    ax_phi.set_ylabel(r"$V$ [V]", color="r")
    ax[1].set_ylabel(r"$E$ [V/m]")
    ax[1].set_xticklabels([])
    ax[1].xaxis.set_tick_params(which="both", size=5, width=1.5, labelsize=13)
    ax[1].yaxis.set_tick_params(which="both", size=5, width=1.5, labelsize=13)

    ax_b = ax[2].twinx()
    ax[2].plot(x_center * 100, P[1, :], linewidth=1.8, markersize=3)
    ax_b.plot(x_center * 100, B, "r:", linewidth=0.7, markersize=3)
    ax_b.set_yticklabels([])
    # ax[2].set_xlabel(r'$x~[m]$')
    ax[2].set_ylabel(r"$n_i$ [m$^{-3}$]")
    # ax[2].xaxis.set_tick_params(which='both', size=0, width=1.5, labelsize=0)
    ax[2].set_xticklabels([])
    ax[2].yaxis.set_tick_params(which="both", size=5, width=1.5, labelsize=13)
    # ax[1].legend(loc = 'lower center', fontsize = 12)

    max_ni = 1e17
    if max(P[1, :]) > 1e18:
        max_ni = max(P[1, :]) * 1.5
    elif max(P[1, :]) > 1e17:
        max_ni = 1e18
    else:
        max_ni = 1e17

    ax[2].set_ylim([0, max_ni])

    ax_b = ax[3].twinx()
    ax[3].plot(x_center * 100, P[3, :], linewidth=1.8, markersize=3)
    ax_b.plot(x_center * 100, B, "r:", linewidth=0.7, markersize=3)
    ax_b.set_yticklabels([])
    # ax[3].set_xlabel(r'$x~[m]$')
    ax[3].set_ylabel(r"$T_e$ [eV]")
    # ax[3].xaxis.set_tick_params(which='both', size=0, width=1.5, labelsize=0)
    ax[3].set_xticklabels([])
    ax[3].yaxis.set_tick_params(which="both", size=5, width=1.5, labelsize=13)
    # ax[3].legend(loc = 'lower center', fontsize = 12)
    # f.suptitle('t = ', fontname = 'Times New Roman',fontsize=16)

    ax_b = ax[4].twinx()
    ax[4].plot(x_center * 100, P[2, :] / 1000, linewidth=1.8, markersize=3)
    ax[4].plot(
        x_center * 100,
        np.sqrt(phy_const.e * P[3, :] / (131.293 * phy_const.m_u)) / 1000.0,
        "g--",
        linewidth=1.8,
        markersize=3,
    )
    ax_b.plot(x_center * 100, B, "r:", linewidth=0.7, markersize=3)
    ax_b.set_yticklabels([])
    # ax[4].set_xlabel(r'$x~[m]$')
    ax[4].set_ylabel(r"$v_i$ [km/s]")
    # ax[4].xaxis.set_tick_params(which='both', size=0, width=1.5, labelsize=0)
    # ax[4].set_xticklabels([])
    # ax[4].yaxis.set_tick_params(which='both', size=5, width=1.5, labelsize=13)
    # ax[4].legend(loc="lower center", fontsize=12)

    ax[5].plot(x_center * 100, P[4, :] / 1000, linewidth=1.8, markersize=3)
    ax[5].set_ylabel(r"$v_e$ [km/s]")
    # ax[4].legend(loc = 'lower center', fontsize = 12)
    title = "time = " + str(round(t / 1e-6, 4)) + "$\mu$s"
    # f[0].suptitle(title, y=1.05)

    ax[6].plot(time / 1e-3, Current)
    ax[6].set_ylabel(r"Current [A]")
    ax[6].set_xlabel(r"time [ms]")
    ax[6].plot(time[file_number] / 1e-3, Current[file_number], "ro", markersize=10)
    ax[6].grid(True)
    ax[6].set_xlim([0, time[-1] / 1e-3])
    ax[6].set_xlim([0, 1])

    for axis in ax:
        axis.grid(True)
        # axis.get_legend().remove()

    # ax[0].legend(fontsize=10, loc="lower right")
    plt.tight_layout()

    plt.subplots_adjust(wspace=0.4, hspace=0.4)
    gls.update(hspace=0.6)

    fig.align_labels()

    for axx in ax:
        # axx.label_outer()
        # axx.xaxis.set_tick_params(which='both', size=5, width=1.5, labelsize=13)
        # axx.yaxis.set_tick_params(which='both', size=5, width=1.5, labelsize=13)
        axx.grid(True, which="both", linestyle=":", alpha=0.5)
        if axx != ax[6]:
            axx.set_xlim([0, LENGTH * 100])
            if axx == ax[5] or axx == ax[4]:
                axx.set_xlabel(r"$x$~[cm]")
        else:
            axx.set_xlabel(r"$t$~[ms]")
            axx.set_xlim([0, time[-1] / 1e-3])

    # plt.savefig(ResultsFigs+"/MacroscopicVars_New_"+str(i_save)+".png", bbox_inches='tight')
    # plt.close()

In [ ]:
with open(files[-1], "rb") as f:
    [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)
ng = P[0,:]
ni = P[1,:]
ui = P[2,:]
Te = P[3,:]
ve = P[4,:]

print("Te_Cathode = ", Te_Cath, Te[-1])
delta_Te = Te[-1] - Te[-2]
Te_extrap = Te[-1] + delta_Te
print("Te_extrap = ", Te_extrap)
plt.figure()
plt.plot(x_center*100, Te)
plt.axhline(y=Te_Cath, color='r', linestyle='--')
plt.xlabel(r'$x$ [cm]', fontsize=18)
plt.ylabel(r'$T_e$ [eV]', fontsize=18)
plt.title('Electron temperature', fontsize=18)
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.xlim(0, LENGTH*100)
plt.ylim(0, 1.1*Te.max())
plt.tight_layout()



In [ ]:
def compute_Kel(Te):
    """This function calculates the ionization rate"""
    # Polynomial coefficients
    c0 = -3.04474930e+01
    c1 = 1.89683694e+00
    c2 = -6.63807968e-01
    c3 = 9.37924042e-03
    c4 = 2.19404998e-02
    c5 = -2.27126387e-03

    # Compute the natural logarithm of Te
    log_Te = np.log(Te)

    # Manually evaluate the polynomial using Horner's method (unrolled loop)
    result = c5
    result = result * log_Te + c4
    result = result * log_Te + c3
    result = result * log_Te + c2
    result = result * log_Te + c1
    result = result * log_Te + c0

    # Return the exponential of the polynomial result
    return np.exp(result)

def compute_Kiz(Te):
    # Xenon ionization
    K0      = 1.18122959e-13
    epsilon = 12.13
    A       = 1.29330521e-01
    B       = 1.00068880e-02
    C       = 6.97445869e-01

    return K0*np.exp(-epsilon/Te)*(np.log(1 + A*Te + B*Te**2))**C

In [ ]:
import pandas as pd

alpha_B = (
    np.ones(NBPOINTS) * alpha_B1
)  # Anomalous transport coefficient inside the thruster
alpha_B = np.where(
    x_center < L0, alpha_B, alpha_B2
)  # Anomalous transport coefficient in the plume
alpha_B_smooth = np.copy(alpha_B)
wce = phy_const.e * B / m  # electron cyclotron frequency

ng_mean = np.zeros_like(P[0, :])
ni_mean = np.zeros_like(P[1, :])
ui_mean = np.zeros_like(P[2, :])
Te_mean = np.zeros_like(P[3, :])
ve_mean_x = np.zeros_like(P[4, :])
ve_mean_y = np.zeros_like(P[4, :])
E_mean = np.zeros_like(P[3, :])
phi_mean = np.zeros_like(P[3, :])
flux_mean = np.zeros_like(P[3, :])
flux_i_mean = np.zeros_like(P[3, :])
flux_e_mean = np.zeros_like(P[3, :])
nu_m_mean = np.zeros_like(P[3, :])
Siz_mean = np.zeros_like(P[3, :])


PTS = 1000
t_vec = []

for ind in range(PTS):
    with open(files[ind - PTS - 100], "rb") as f:
        [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)
        t_vec.append(t)
        ng = P[0, :]
        ni = P[1, :]
        ui = P[2, :]
        Te = P[3, :]
        ve = P[4, :]
        E = compute_E(P, J)
        phi = compute_phi(P, J)
        Kel = compute_Kel(Te)  # Electron - neutral  collision rate
        Kiz = compute_Kiz(Te)  # Electron - neutral ionization rate

        ############################
        #       Wall Collisions    #
        ############################
        sigma = 0.207 * Te ** (0.549)
        sigma_scl = 1.0 - 8.3 * np.sqrt(m / M)
        sigma[sigma > sigma_scl] = sigma_scl
        nu_iw = 2 * 0.5 * (1.0 / (R2 - R1)) * np.sqrt(phy_const.e * Te / M)
        index_L0 = np.argmax(x_center > L0)
        nu_iw[index_L0:] = 0.0
        nu_ew = nu_iw / (1 - sigma)  # Electron - wall collision rate

        nu_m = (
            ng * Kel + alpha_B * wce
        )  # Electron momentum - transfer collision frequency

        if ind > 0:
            ng_mean += ng * (t_vec[ind] - t_vec[ind - 1])
            ni_mean += ni * (t_vec[ind] - t_vec[ind - 1])
            ui_mean += ui * (t_vec[ind] - t_vec[ind - 1])
            Te_mean += Te * (t_vec[ind] - t_vec[ind - 1])
            ve_mean_x += ve * (t_vec[ind] - t_vec[ind - 1])
            ve_mean_y += ve / (nu_m / wce) * (t_vec[ind] - t_vec[ind - 1])
            E_mean += E * (t_vec[ind] - t_vec[ind - 1])
            phi_mean += phi * (t_vec[ind] - t_vec[ind - 1])
            flux_i_mean += (ui * ni) * (t_vec[ind] - t_vec[ind - 1])
            flux_e_mean += (-ve * ni) * (t_vec[ind] - t_vec[ind - 1])
            flux_mean += (ui * ni - ve * ni) * (t_vec[ind] - t_vec[ind - 1])
            nu_m_mean += nu_m * (t_vec[ind] - t_vec[ind - 1])
            Siz_mean += ng * ni * Kiz * (t_vec[ind] - t_vec[ind - 1])

ng_mean /= t_vec[-1] - t_vec[0]
ni_mean /= t_vec[-1] - t_vec[0]
ui_mean /= t_vec[-1] - t_vec[0]
Te_mean /= t_vec[-1] - t_vec[0]
ve_mean_x /= t_vec[-1] - t_vec[0]
ve_mean_y /= t_vec[-1] - t_vec[0]
E_mean /= t_vec[-1] - t_vec[0]
phi_mean /= t_vec[-1] - t_vec[0]
flux_i_mean /= t_vec[-1] - t_vec[0]
flux_e_mean /= t_vec[-1] - t_vec[0]
flux_mean /= t_vec[-1] - t_vec[0]
nu_m_mean /= t_vec[-1] - t_vec[0]
Siz_mean /= t_vec[-1] - t_vec[0]


#############################
#       Compute the rates   #
#############################


Kel = compute_Kel(Te_mean)  # Electron - neutral  collision rate
Kiz = compute_Kiz(Te_mean)  # Electron - neutral ionization rate

############################
#       Wall Collisions    #
############################
sigma = 0.207 * Te_mean ** (0.549)
sigma_scl = 1.0 - 8.3 * np.sqrt(m / M)
sigma[sigma > sigma_scl] = sigma_scl
nu_iw = 2 * 0.5 * (1.0 / (R2 - R1)) * np.sqrt(phy_const.e * Te_mean / M)
index_L0 = np.argmax(x_center > L0)
nu_iw[index_L0:] = 0.0
nu_ew = nu_iw / (1 - sigma)  # Electron - wall collision rate

alpha_B = (
    np.ones(NBPOINTS) * alpha_B1
)  # Anomalous transport coefficient inside the thruster
alpha_B = np.where(
    x_center < L0, alpha_B, alpha_B2
)  # Anomalous transport coefficient in the plume
alpha_B_smooth = np.copy(alpha_B)

# smooth between alpha_B1 and alpha_B2
for index in range(10, NBPOINTS - 9):
    alpha_B_smooth[index] = np.mean(alpha_B[index - 10 : index + 10])
alpha_B = alpha_B_smooth

nu_m = ng_mean * Kel + alpha_B * wce  # Electron momentum - transfer collision frequency

pand = pd.DataFrame(
    {
        "xx_center": x_center,
        "temperature": P[3, :],
        "ng": P[0, :],
        "ni": P[1, :],
        "ui": P[2, :],
        "ve": P[4, :],
        "E": E,
        "phi": phi,
        "B": B,
        "nu_m": nu_m,
        "Siz": ng * ni * Kiz,
    }
)
pand = pd.DataFrame(
    {
        "xx_center": x_center,
        "temperature": Te_mean,
        "ng": ng_mean,
        "ni": ni_mean,
        "ui": ui_mean,
        "vex": ve_mean_x,
        "vey": ve_mean_y,
        "E": E_mean,
        "phi": phi_mean,
        "B": B,
        "nu_m": nu_m_mean,
        "Siz": Siz_mean,
        "flux": flux_mean,
        "flux_i": flux_i_mean,
        "flux_e": flux_e_mean,
    }
)

pand.to_csv(Results + "/values_fluid_unstat.csv", index=False)
print("Fluid values saved to CSV file:", Results + "/values_fluid_unstat.csv")
# os.system("ffmpeg -r 10 -i "+ResultsFigs+"/MacroscopicVars_New_%d.png -vcodec mpeg4 -y -vb 20M "+ResultsFigs+"Evolution.mp4")

In [ ]:
plt.figure()
gls = plt.GridSpec(3, 2)
ax0 = plt.subplot(gls[0, 0])
ax0.plot(x_center * 100, ng_mean, label='ng')
ax0.plot(x_center * 100, ng, label='ng', ls = '--')
ax0.set_xlabel(r'$x$ [cm]', fontsize=18)
ax0.set_ylabel(r'$n_g$ [m$^{-3}$]', fontsize=18)
ax0.set_title('Neutral density', fontsize=18)
ax0.grid(True, which='both', linestyle=':', alpha=0.5)
ax0.set_xlim(0, LENGTH * 100)
ax0.set_ylim(0, 1.1 * ng_mean.max())
ax1 = plt.subplot(gls[0, 1])
ax1.plot(x_center * 100, ni_mean, label='ni')
ax1.plot(x_center * 100, ni, label='ni', ls = '--')
ax1.set_xlabel(r'$x$ [cm]', fontsize=18)
ax1.set_ylabel(r'$n_i$ [m$^{-3}$]', fontsize=18)
ax1.set_title('Ion density', fontsize=18)
ax1.grid(True, which='both', linestyle=':', alpha=0.5)
ax1.set_xlim(0, LENGTH * 100)
ax1.set_ylim(0, 1.1 * ni_mean.max())
ax2 = plt.subplot(gls[1, 0])
ax2.plot(x_center * 100, ui_mean / 1000, label='ui')
ax2.plot(x_center * 100, ui / 1000, label='ui', ls = '--')
ax2.set_xlabel(r'$x$ [cm]', fontsize=18)
ax2.set_ylabel(r'$u_i$ [km/s]', fontsize=18)
ax2.set_title('Ion velocity', fontsize=18)
ax2.grid(True, which='both', linestyle=':', alpha=0.5)
ax2.set_xlim(0, LENGTH * 100)
ax2.set_ylim(0, 1.1 * ui_mean.max() / 1000)
ax3 = plt.subplot(gls[1, 1])
ax3.plot(x_center * 100, Te_mean, label='Te')
ax3.plot(x_center * 100, Te, label='Te', ls = '--')
ax3.set_xlabel(r'$x$ [cm]', fontsize=18)
ax3.set_ylabel(r'$T_e$ [eV]', fontsize=18)
ax3.set_title('Electron temperature', fontsize=18)
ax3.grid(True, which='both', linestyle=':', alpha=0.5)
ax3.set_xlim(0, LENGTH * 100)
ax4 = plt.subplot(gls[2, 0])
ax4.plot(x_center * 100, ve_mean_y / 1000, label='ve')
# ax4.plot(x_center * 100, ve / 1000, label='ve', ls = '--')
ax4.set_xlabel(r'$x$ [cm]', fontsize=18)
ax4.set_ylabel(r'$v_e$ [km/s]', fontsize=18)
ax4.set_title('Electron velocity', fontsize=18)
ax4.grid(True, which='both', linestyle=':', alpha=0.5)
ax4.set_xlim(0, LENGTH * 100)
ax5 = plt.subplot(gls[2, 1])
ax5.plot(x_center * 100, E_mean, label='E')
ax5.plot(x_center * 100, E, label='E', ls = '--')
ax5.set_xlabel(r'$x$ [cm]', fontsize=18)
ax5.set_ylabel(r'$E$ [V/m]', fontsize=18)
ax5.set_title('Electric field', fontsize=18)
ax5.grid(True, which='both', linestyle=':', alpha=0.5)
ax5.set_xlim(0, LENGTH * 100)
ax5.set_ylim(0, 1.1 * E_mean.max())
# ax6 = plt.subplot(gls[2, 1])


        

In [ ]:
plt.figure()
# plt.plot(x_center * 100,ve_mean_x * wce / nu_m_mean, label='phi')
plt.plot(x_center * 100,Siz_mean, label='phi')

In [ ]:
plt.figure()
plt.plot(x_center * 100, ve_mean * ni_mean, label='mean', c='orange')
plt.plot(x_center * 100, ve * ni, label='point', ls = '--', c='blue')
plt.plot(x_center * 100, ui_mean * ni_mean, ls = '-', c='orange')
plt.plot(x_center * 100, ui * ni, ls = '--', c='blue')
plt.plot(x_center * 100, ui_mean * ni_mean - ve_mean * ni_mean, ls = ':', c='orange')
plt.plot(x_center * 100, ui * ni - ve * ni, ls = ':', c='blue')
plt.plot(x_center * 100, flux_mean, label='flux_mean', ls = '-.', c='green')

plt.xlabel(r'$x$ [cm]', fontsize=18)
plt.ylabel(r'$n_i v_i - n_e v_e$ [m$^{-2}$s$^{-1}$]', fontsize=18)
plt.title('Ion and electron fluxes', fontsize=18)
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.xlim(0, LENGTH * 100)
# plt.ylim(0, 1.1 * (ui_mean * ni_mean).max())
plt.legend()

In [ ]:
MM = 131.293 * phy_const.m_u  # Xenon mass in kg
A_ch = np.pi * (R2**2 - R1**2)  # Cross-sectional area of the channel in m^2

print("Mass flow rate = ", mdot, " kg/s")
print("A_ch = ", A_ch, " m^2", "R2 = ", R2, " m", "R1 = ", R1, " m")
thrust = ni[-1] * ui[-1] ** 2 * MM * A_ch * 1e3
print("Thrust = ", thrust, " mN")
Isp = thrust * 1e-3 / (mdot * phy_const.g)
print("Isp = ", Isp, " s")
Id = Current[-1]
print("Id = ", Id, " A")

## Multicase results processing

In [ ]:
def calculate_eng_params(Results, ind):
    ResultsFigs = Results + "/Figs"
    ResultsData = Results + "/Data"
    if not os.path.exists(ResultsFigs):
        os.makedirs(ResultsFigs)

    ResultConfig = Results + "/Configuration.cfg"
    configFile = ResultConfig
    config = configparser.ConfigParser()
    config.read(configFile)

    physicalParameters = config["Physical Parameters"]
    mdot = float(physicalParameters["Mass flow"])  # Mass flow rate of propellant
    Te_Cath  = float(physicalParameters['Temperature Cathode'])    

    # open all the files in the directory and sort them to do the video in order
    files = glob.glob(ResultsData + "/*.pkl")
    files.sort(key=os.path.getmtime)
    files = files[:]

    # with open(files[-1], "rb") as f:
    #     [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)
    PTS = 750
    t_vec = []
    J_vec = []

    for ind in range(PTS):
        with open(files[ind-PTS], "rb") as f:
            [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)
            t_vec.append(t)
            J_vec.append(J)
            ng = P[0,:]
            ni = P[1,:]
            ui = P[2,:]
            Te = P[3,:]
            ve = P[4,:]
            if ind == 0:
                ng_mean = np.zeros_like(ng)
                ni_mean = np.zeros_like(ni)
                ui_mean = np.zeros_like(ui)
                Te_mean = np.zeros_like(Te)
                ve_mean = np.zeros_like(ve)
                j_mean = 0
            if ind > 0:
                ng_mean += ng * (t_vec[ind] - t_vec[ind-1])
                ni_mean += ni * (t_vec[ind] - t_vec[ind-1])
                ui_mean += ui * (t_vec[ind] - t_vec[ind-1])
                Te_mean += Te * (t_vec[ind] - t_vec[ind-1])
                ve_mean += ve * (t_vec[ind] - t_vec[ind-1])
                j_mean += J * (t_vec[ind] - t_vec[ind-1])
    ng_mean /= (t_vec[-1] - t_vec[0])
    ni_mean /= (t_vec[-1] - t_vec[0])
    ui_mean /= (t_vec[-1] - t_vec[0])
    Te_mean /= (t_vec[-1] - t_vec[0])
    ve_mean /= (t_vec[-1] - t_vec[0])
    j_mean = (j_mean) / (t_vec[-1] - t_vec[0])


    # interpolate J_vec
    J_interp = np.interp(np.linspace(t_vec[0], t_vec[-1], 500), t_vec[:], J_vec[:])
    J_STD = np.std(J_interp)
    # print(f"J mean: {j_mean:.2f} A/m^2, J STD: {J_STD:.2f} A/m^2")
    # plt.figure()
    # plt.plot(np.array(t_vec)*1e3, J_vec, label='J')
    # plt.plot(np.linspace(t_vec[0], t_vec[-1], 500) * 1e3, J_interp)
    ng = ng_mean
    ni = ni_mean
    ui = ui_mean
    Te = Te_mean
    ve = ve_mean

    # Calculate the thrust and specific impulse
    thrust = ni[-1] * ui[-1] ** 2 * MM * A_ch * 1e3
    Isp = thrust * 1e-3 / (mdot * phy_const.g)
    print(f"{ind} mdot: {mdot*1e6:.2f} mg/s, mean between {t_vec[0]*1e3:.3f} and {t_vec[-1]*1e3:.3f} ms, I = {j_mean:.2f} A and J_STD = {J_STD:.2f} A")
    return thrust, Isp, j_mean, J_STD, mdot, V - J * 10, t, Te_Cath

In [ ]:
from tqdm import tqdm
import pandas as pd
thrust_vec = []
Isp_vec    = []
J_vec     = []
J_std_vec = []
mdot_vec   = []
V_vec     = []
Te_Cath_vec = []

MM = 131.293 * phy_const.m_u  # Xenon mass in kg
A_ch = np.pi * (R2**2 - R1**2)  # Cross-sectional area of the channel in m^2

# for ind in tqdm(range(6,24)): 
for ind in range(6,24): 
# for ind in range(22,23): 

    # Results     = "Results_SPT_2/test_"+str(ind)+"/"
    Results     = "Results_SPT_h03/test_"+str(ind)+"/" 
    # Results     = "Results_SPT_h03_1/test_"+str(ind)+"/"  # CASE WITHOUT BALLAST
    thrust, Isp, J, J_std, mdot, VV, time_fin, Te_Cath = calculate_eng_params(Results, ind)
    if time_fin > .8e-3:
        # Save the results
        thrust_vec.append(thrust)
        Isp_vec.append(Isp)
        J_vec.append(J)
        J_std_vec.append(J_std)
        mdot_vec.append(mdot)
        V_vec.append(VV)
        Te_Cath_vec.append(Te_Cath)
    # else:
    #     thrust_vec.append(0)
    #     Isp_vec.append(0)
    #     J_vec.append(0)
    #     mdot_vec.append(mdot)
    #     V_vec.append(0)

# Save the results as pd dataframe

df = pd.DataFrame({
    "thrust": thrust_vec,
    "Isp": Isp_vec,
    "J": J_vec,
    "J_std": J_std_vec,
    "mdot": mdot_vec,
    "V": V_vec,
})
# df.to_csv("Results_SPT_h03/eng_params_BIS.csv", index=False)
# Plot the results

print(thrust_vec)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
ax[0].plot(mdot_vec, Isp_vec, 'ro')
ax[0].set_xlabel(r'Mass flow rate [kg/s]', fontsize=18)
ax[0].set_ylabel(r'Specific impulse [s]', fontsize=18)
ax[0].set_title('Isp vs Mass flow rate', fontsize=18)
ax[1].plot(mdot_vec, thrust_vec, 'ro')
ax[1].set_xlabel(r'Mass flow rate [kg/s]', fontsize=18)
ax[1].set_ylabel(r'Thrust [mN]', fontsize=18)
ax[1].set_title('Thrust vs Mass flow rate', fontsize=18)
ax[2].plot(mdot_vec, J_vec, 'ro')
ax[2].errorbar(mdot_vec, J_vec, yerr=J_std_vec, fmt='ro', capsize=5, label='Isp with std dev')
ax[2].set_xlabel(r'Mass flow rate [kg/s]', fontsize=18)
ax[2].set_ylabel(r'Current [A]', fontsize=18)
ax[2].set_title('Current vs Mass flow rate', fontsize=18)
plt.tight_layout()
# plt.savefig(ResultsFigs+"/Eng_params.png", bbox_inches='tight')b

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
ax[0].plot(Te_Cath_vec, Isp_vec, 'ro')
ax[0].set_xlabel(r'Mass flow rate [kg/s]', fontsize=18)
ax[0].set_ylabel(r'Specific impulse [s]', fontsize=18)
ax[0].set_title('Isp vs Mass flow rate', fontsize=18)
ax[1].plot(Te_Cath_vec, thrust_vec, 'ro')
ax[1].set_xlabel(r'Mass flow rate [kg/s]', fontsize=18)
ax[1].set_ylabel(r'Thrust [mN]', fontsize=18)
ax[1].set_title('Thrust vs Mass flow rate', fontsize=18)
ax[2].plot(Te_Cath_vec, J_vec, 'ro')
ax[2].set_xlabel(r'Mass flow rate [kg/s]', fontsize=18)
ax[2].set_ylabel(r'Current [A]', fontsize=18)
ax[2].set_title('Current vs Mass flow rate', fontsize=18)
plt.tight_layout()
# plt.savefig(ResultsFigs+"/Eng_params.png", bbox_inches='tight')

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(mdot_vec, mdot_vec, 'ro')
plt.subplot(1, 2, 2)
plt.plot(mdot_vec, V_vec, 'ro')


In [ ]:
def compute_Kel(Te):
    """This function calculates the ionization rate"""
    # Polynomial coefficients
    c0 = -3.04474930e01
    c1 = 1.89683694e00
    c2 = -6.63807968e-01
    c3 = 9.37924042e-03
    c4 = 2.19404998e-02
    c5 = -2.27126387e-03

    # Compute the natural logarithm of Te
    log_Te = np.log(Te)

    # Manually evaluate the polynomial using Horner's method (unrolled loop)
    result = c5
    result = result * log_Te + c4
    result = result * log_Te + c3
    result = result * log_Te + c2
    result = result * log_Te + c1
    result = result * log_Te + c0

    # Return the exponential of the polynomial result
    return np.exp(result)

def compute_Kiz(Te):
    # Xenon ionization
    K0 = 1.18122959e-13
    epsilon = 12.13
    A = 1.29330521e-01
    B = 1.00068880e-02
    C = 6.97445869e-01

    # Ensure Te is an array for element-wise operations
    Te = np.asarray(Te)

    # Debugging statements
    if np.any(Te < 0):
        print("Te contains negative values:", Te)
    
    if np.any(np.isnan(Te)):
        print("Te contains NaN values:", Te)

    log_arg = 1 + A * Te + B * Te**2
    if np.any(log_arg <= 0):  # Ensuring no negative values inside log
        print("Invalid log argument encountered:", log_arg)

    if np.any(np.isnan(log_arg)):
        print("log_arg contains NaN values:", log_arg)

    log_term = np.log(log_arg)

    if np.any(np.isnan(log_term)):
        print("NaN encountered in log computation:", log_term)
        print("Corresponding log_arg values:", log_arg[np.isnan(log_term)])

    return K0 * np.exp(-epsilon / Te) * (log_term) ** C

def computeEpsilonLoss(Te):
    def computeKprocess(K0, epsilon, A, B, C):
        arg = 1 + A * Te + B * Te**2
        if np.any(arg <= 1):
            arg = 1.0
        if (np.any(np.isnan( K0 * np.exp(-epsilon / Te) * (np.log(arg) ** C)))):
            print("NaN encountered in computeKprocess:", K0, epsilon, A, B, C)
            print("Te value:", Te)
            print("Corresponding log argument:", arg)
            print("Corresponding log term:", np.log(arg))
        return K0 * np.exp(-epsilon / Te) * (np.log(arg)) ** C

    K_iz = computeKprocess(
        1.18122959e-13, 12.13, 1.29330521e-01, 1.00068880e-02, 6.97445869e-01
    )
    K_ex1 = computeKprocess(
        2.37016128e-14, 8.315, 7.99682247e-02, -5.91358673e-04, 4.51997276e-01
    )
    K_ex2 = computeKprocess(
        9.02951389e-15, 9.447, 3.12421531e00, -3.01100074e-02, 5.59327899e-01
    )
    K_ex3 = computeKprocess(
        1.66394517e-14, 9.917, 2.83412200e00, -2.66987222e-02, 6.98378384e-01
    )
    K_ex4 = computeKprocess(
        7.64651071e-15, 11.70, 7.35828827e-01, -5.08912904e-03, 1.39724961e00
    )

    return (
        12.13
        + (K_ex1 * 8.315 + K_ex2 * 9.447 + K_ex3 * 9.917 + K_ex4 * 11.70) / K_iz
        + 3 * m / M * compute_Kel(Te) * Te / K_iz
    )

In [ ]:
epsilon_val = []
for Te_val in np.arange(1, 150., 0.1):
    epsilon_val.append(computeEpsilonLoss(Te_val))

eps_c = np.loadtxt(
   "/home/petronio/Nextcloud/code/projet_code_stationnaire_EA/library_code/coll_data/EpsC_xenon.csv",
    delimiter=",",
)
# Perform exponential interpolation
x = eps_c[:, 0]
y = eps_c[:, 1]
# Create an exponential interpolation function
from scipy.interpolate import interp1d
interp_func_eps_c = interp1d(x, np.log(y), kind="linear", fill_value="extrapolate")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, 150., 0.1), epsilon_val, marker="v")
plt.plot(
    eps_c[:, 0], eps_c[:, 1], "ro", label="Experimental data", markersize=3
)
plt.xlabel(r"$T_e$ [eV]", fontsize=18)
plt.yscale("log")
plt.ylabel(r"$\epsilon_{loss}$ [eV]", fontsize=18)
plt.ylim(1e0, 1e3)